# Deep Web Research Workflow

In this notebook, we will consider a workflow that does a deep web research. Given a research request, the workflow will perform the following steps:

1. Generate research queries based on the request.
2. Execute the queries using a search tool.
3. Consolidate the results into a knowledge base.
4. Make decisions about further research rounds based on the consolidated knowledge.

## Setup and Imports

As usual, we import all necessary modules and set up the essentials.

In [ ]:
import os
import dotenv
import json
from IPython.display import Image, display
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Literal, Tuple, Annotated
from operator import add
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import START, END, StateGraph
from langgraph.types import Command
from langfuse.langchain import CallbackHandler

# Load environment variables
dotenv.load_dotenv()

# Initialize Langfuse for tracing
langfuse_handler = CallbackHandler()

# Initialize the chat model
model = init_chat_model(
    os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"],
    max_tokens=8096
)

# Research Knowledge Entries -- Our data structure for storing research findings

Before we define the workflow itself, we specify the data structure we will use to store our research findings. This structure will help us organize and manage the information we gather during the research process.

In [ ]:
class ResearchKnowledgeEntry(BaseModel):
    """A knowledge entry. Contains factual information gathered from the research."""

    source_url: str = Field(
        ...,
        description="The URL of source of the knowledge entry. The book, article, website, etc. where the information was found. Title and author."
    )

    content: str = Field(
        ...,
        description="The content of the knowledge entry in a condensed form. It should be a concise summary of the information found at the source."
    )

    quote: str = Field(
        ...,
        description="The verbatim quote from the original text."
    )


class ResearchKnowledgeEntryList(BaseModel):
    """A list of knowledge entries."""

    entries: List[ResearchKnowledgeEntry] = Field(
        default_factory=list,
        description="A list of knowledge entries."
    )

Both classes should be rather self-explanatory, with clear field names and descriptions. The `ResearchKnowledgeEntry` class represents a single piece of knowledge gathered during the research process, while the `ResearchKnowledgeEntryList` class is simply a container for multiple knowledge entries.

## The Research Workflow

With our data structures in place, we can now outline the research workflow. This workflow will guide the process of gathering, organizing, and synthesizing information from various sources to address the research request.

We start with the `ResearchState` class, which holds all the necessary information about the current state of the research process. This includes the research request, the target number of tokens for the output, and the list of research queries generated from the request.

In [ ]:
class ResearchState(BaseModel):
    """State for the research workflow."""

    research_request: str = Field(
        default="",
        description="The research request to be fulfilled."
    )

    target_tokens: int = Field(
        default=512,
        description="The target number of tokens for the research output."
    )

    research_queries: Annotated[List[Dict[str, str]], add] = Field(
        default_factory=list,
        description="A list of research queries generated from the research request."
    )

    research_queries_results: List[Dict[str, Any]] = Field(
        default_factory=list,
        description="A list of results for each research query."
    )

    consolidated_knowledge: Annotated[List[ResearchKnowledgeEntry], add] = Field(
        default_factory=list,
        description="A secuence of knowledge items"
    )

The field `research_request` is the starting point for the research process. The field `consolidated_knowledge` is where we will store the final synthesized knowledge gathered from the research queries and their results.

Before we define our nodes, we introduce a special tool. `TavilySearch` is a powerful search tool that will help us find relevant information from various sources during the research process. We will use this tool to perform searches based on the research queries generated from the initial research request.

In [ ]:
def get_tavily_search_tool() -> TavilySearch:
    return TavilySearch(
        max_results=5,
        topic="general",
        include_raw_content=True,
        include_domains=None,
        exclude_domains=None
    )

Now we can define our first node. `query_generation_node` will generate the initial query and kick off the research process.

In [ ]:
async def query_generation_node(state: ResearchState, config) -> ResearchState:

    # Log the node.
    print("Starting query generation node...")

    # Create the system message.
    system_message_components = [
        "You are an expert researcher.",
        "You will do content research based on the persona.",
        "You have access to tavily websearch.",
    ]
    system_message = SystemMessage(
        content="\n".join(system_message_components)
    )

    # Create a human message that asks for the content generation.
    human_message_components = [
        f"Here is a research request:\n\n```\n{state.research_request}\n```\n",
        f"Here are the research queries generated so far:\n\n```\n{state.research_queries}\n```\n",
        "Generate a tavily search query, that retrieves relevant documents based on the research request."
    ]
    human_message = HumanMessage(
        content="\n".join(human_message_components)
    )
    
    # Create the chat model with tools.
    chat_model_with_tools = model.bind_tools([get_tavily_search_tool()])

    # Invoke the chat model with the messages.
    print("Invoking the chat model to generate tavily search queries...")
    response = await chat_model_with_tools.ainvoke(
        [system_message, human_message]
    )
    if not response.tool_calls or len(response.tool_calls) == 0:
        raise Exception("No tool calls were made by the model.")
    if len(response.tool_calls) > 1:
        raise Exception("Multiple tool calls were made by the model.")
    query = response.tool_calls[0]["args"]

    # Done.
    return {
        "research_queries": [query]
    }

The next node is `knowledge_research_node`. This one will execute the last query and store the results in the state.

In [ ]:
async def knowledge_research_node(state: ResearchState, config) -> ResearchState:

    # Log the node.
    print("Starting knowledge research node...")

    # Get the last query.
    query = state.research_queries[-1]

    search_results = await get_tavily_search_tool().ainvoke(
        query
    )
    search_results = search_results["results"]
    print(f"Found {len(search_results)} search results.")
    return {
        "research_queries_results": search_results
    }

The `knowledge_consolidation_node` is responsible for taking the results of the research queries and consolidating them into a coherent set of knowledge items. This involves processing each search result, extracting relevant information, and updating the state with the new knowledge.

In [ ]:
async def knowledge_consolidation_node(state: ResearchState, config) -> Command[Literal["knowledge_consolidation_node", "another_round_decision_node"]]:

    # Finish when we have processed everything.
    if len(state.research_queries_results) == 0:
        print("All research query results have been processed.")
        return Command(
            goto="another_round_decision_node",
            update={}
        )

    # Log the node.
    print(f"Starting knowledge consolidation node with {len(state.research_queries_results)} results left to process.")

    # Get the first search result.
    first_search_result = state.research_queries_results.pop(0)

    # Create the system message.
    system_message_components = [
        "You are an expert researcher.",
        "Given a couple of vector database query results, you will turn them into consolidated knowledge items.",
        f"Here is the schema for vector database query results: {ResearchKnowledgeEntryList.model_json_schema()}"
    ]
    system_message = SystemMessage(
        content="\n".join(system_message_components)
    )

    # Create a human message that asks for the content generation.
    human_message_components = []
    human_message_components.append(f"Here is a tavily search query result:\n\n```\n{first_search_result}\n```\n")
    for knowledge_entry in state.consolidated_knowledge:
        human_message_components.append(f"Here is a consolidated knowledge item:\n\n```\n{knowledge_entry}\n```\n")
    human_message_components.append(f"Here is the research request: `{state.research_request}`.")
    human_message_components.append("Based on this, create a list of new knowledge items, we will add to the collection.")
    human_message_components.append("Make sure to include all relevant information from the query results that is relevant to the research request. Discard any irrelevant information.")
    human_message = HumanMessage(
        content="\n".join(human_message_components)
    )

    # Create the chat model with tools.
    chat_model_with_structured_output = model.with_structured_output(ResearchKnowledgeEntryList)
    consolidated_knowledge_items = await chat_model_with_structured_output.ainvoke(
        [system_message, human_message]
    )
    consolidated_knowledge_items = consolidated_knowledge_items.entries

    assert len(consolidated_knowledge_items) > 0

    # Done.
    return Command(
        goto="knowledge_consolidation_node",
        update={
            "research_queries_results": state.research_queries_results,
            "consolidated_knowledge": consolidated_knowledge_items
        }
    )

The final node is `knowledge_consolidation_node`. This one will decide whether to do another round of research based or not.

In [ ]:
@tool
def stop_tool(reason: str) -> None:
    """
    Stops the workflow.

    Args:
        reason (str): The reason for stopping the workflow.
    """
    pass

async def another_round_decision_node(state: ResearchState, config) -> Command[Literal["knowledge_research_node", END]]:

    # Log the node.
    print("Deciding whether to do another round of research...")

    # Count the tokens.
    tokens = 0
    for knowledge_entry in state.consolidated_knowledge:
        characters = len(knowledge_entry.content) + len(knowledge_entry.quote)
        tokens += characters // 4

    # Invoke the decision-making process.
    # Create the system message.
    system_message_components = []
    system_message_components += ["You are an expert researcher."]
    system_message_components += [
        "Given a request, a couple of tavily search queries and their results, we should do another round of research.",
    ]
    system_message = SystemMessage(
        content="\n".join(system_message_components)
    )

    # Create human message.
    human_message_components = []
    human_message_components += [f"Here are the past tavily search queries: {state.research_queries}"]
    for knowledge_item in state.consolidated_knowledge:
        human_message_components += [f"Here is a consolidated knowledge item: `{knowledge_item}`"]
    human_message_components += [f"Here is the request: `{state.research_request}`"]
    human_message_components += [f"We have a target of {state.target_tokens} tokens and currently have {tokens} tokens. Do another round of research if we have not exceeded the target yet. If we have more tokens than the target tokens, stop the workflow by not generating another tool call."]
    human_message_components += [f"Decide whether to do another round of research. If we do not have enough results, generate a new query. Make sure the new query complements the past queries if there were any."]
    human_message = HumanMessage(
        content="\n".join(human_message_components)
    )

    # Create and invoke the model.
    chat_model_with_tools = model.bind_tools([get_tavily_search_tool(), stop_tool])
    response = await chat_model_with_tools.ainvoke(
        [system_message, human_message]
    )
    
    # If we shall do another round or not.
    for tool_call in response.tool_calls:
        if tool_call["name"] == "stop_tool":
            print(f"Stopping the workflow because: {tool_call['args']}")
            return Command(
                goto=END,
                update={}
            )
        if tool_call["name"] == "tavily_search":
            query = tool_call["args"]
            print(f"Generated new query: {query}")
            return Command(
                goto="knowledge_research_node",
                update={
                    "research_queries": [query]
                }
            )
    raise ValueError("No valid tool call found.")

With all the nodes in place, we can now connect them together to form the complete research agent workflow. We add nodes and edges. Note that the `another_round_decision_node` acts as a conditional edge.

In [ ]:
builder = StateGraph(ResearchState)

builder.add_node(query_generation_node)
builder.add_node(knowledge_research_node)
builder.add_node(knowledge_consolidation_node)
builder.add_node(another_round_decision_node)

builder.add_edge(START, "query_generation_node")
builder.add_edge("query_generation_node", "knowledge_research_node")
builder.add_edge("knowledge_research_node", "knowledge_consolidation_node")

graph = builder.compile()

Let us draw the graph before running it.

In [ ]:
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

Now we are ready to run the graph.

In [ ]:
result = await graph.ainvoke(
    {
        "research_request": "Summarize the current state of AI research in the field of natural language processing.",
        "target_tokens": 128
    },
    config={"callbacks": [langfuse_handler]}
)

for result in result["consolidated_knowledge"]:
    print(json.dumps(result.model_dump(), indent=2))

# Done.